# Week 3: MLP on MNIST (PyTorch)

**Goal:** train a multi-layer perceptron on MNIST or Fashion-MNIST; try 2 learning rates and 2 layer sizes.

**Data:** `torchvision.datasets.MNIST`, downloaded automatically into `data/`.

**Result:** _Fill in when done: one line with the main number._

In [ ]:
from pathlib import Path
import random
import numpy as np
import torch

# Project root (works whether the notebook runs from its own folder or the repo root)
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.ToTensor()
train_ds = datasets.MNIST(DATA_DIR, train=True, download=True, transform=transform)
test_ds = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=512)
print(len(train_ds), "train /", len(test_ds), "test")

## 1. Model

In [ ]:
def make_mlp(hidden=128):
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, hidden), nn.ReLU(),
        nn.Linear(hidden, 10),
    )

## 2. Training loop: predict, loss, backprop, update

In [ ]:
def train_epoch(model, dl, opt, loss_fn):
    model.train()
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad()
        loss.backward()
        opt.step()

@torch.no_grad()
def accuracy(model, dl):
    model.eval()
    correct = total = 0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(1) == yb).sum().item()
        total += len(yb)
    return correct / total

## 3. Experiment: 2 learning rates x 2 layer sizes

In [ ]:
results = []
for lr in [1e-2, 1e-3]:
    for hidden in [64, 256]:
        model = make_mlp(hidden).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        for epoch in range(3):
            train_epoch(model, train_dl, opt, nn.CrossEntropyLoss())
        results.append({"lr": lr, "hidden": hidden, "test_acc": accuracy(model, test_dl)})

import pandas as pd
pd.DataFrame(results)

## Results

| Experiment | Setting | Metric | Notes |
|---|---|---|---|
|  |  |  |  |

## Conclusion

_Three lines: what worked best, why you think so, and what you would try next._